In [ ]:
CAMPOS = ["nombre", "ip", "sistema", "ubicacion", "responsable"]
SISTEMAS_VALIDOS = {"linux", "windows", "macos"}

# --- 1. FUNCIÓN DE LECTURA ---
def leer_fichero(ruta_fichero):
    lineas = []
    f = None
    
    try:
        f = open(ruta_fichero, 'r', encoding='utf-8') 
        
        for linea in f:
            linea = linea.strip()
            if linea and not linea.startswith('#'):
                lineas.append(linea)
                
    except FileNotFoundError:
        print(f"Erro: O ficheiro {ruta_fichero} non foi atopado.")
        return [] 
        
    finally:
        if f:
            f.close()
            
    return lineas

# --- 2. FUNCIÓN DE NORMALIZACIÓN ---
def normalizar_linea(linea):
    linea_uniforme = linea.replace(',', ';')
    partes = [p.strip() for p in linea_uniforme.split(';')]
    
    while len(partes) < len(CAMPOS):
        partes.append("")
        
    diccionario_linea = {}
    
    for i, campo in enumerate(CAMPOS):
        valor = partes[i]
        
        if campo == "sistema":
            diccionario_linea[campo] = valor.lower()
        elif campo == "responsable" and valor:
            diccionario_linea[campo] = valor.capitalize()
        else:
            diccionario_linea[campo] = valor
            
    return diccionario_linea

# --- 3A. FUNCIÓN AUXILIAR DE IP ---
def es_ip_valida(ip_str):
    segmentos = ip_str.split('.')
    if len(segmentos) != 4:
        return False
    
    for seg in segmentos:
        if not seg.isdigit(): 
             return False
        if not 0 <= int(seg) <= 255:
            return False
    return True

# --- 3B. FUNCIÓN DE VALIDACIÓN ---
def validar_datos(diccionario_linea):
    datos = diccionario_linea.copy()
    
    if not datos['nombre']:
        return False, "ERROR_NOMBRE_INVALIDO: O nome está baleiro."
        
    if not es_ip_valida(datos['ip']):
        return False, "ERROR_IP_INVALIDA: O enderezo IP non é válido."

    if datos['sistema'] not in SISTEMAS_VALIDOS:
        return False, "ERROR_SISTEMA_INVALIDO: Sistema operativo non permitido."
        
    if not datos['ubicacion'].strip():
        datos['ubicacion'] = None
        
    if datos['responsable'] == "-" or not datos['responsable']:
        datos['responsable'] = None
        
    return True, datos

# --- 4. FUNCIÓN DE ORQUESTACIÓN Y REPORTE ---
def procesar_inventario(ruta_fichero):
    lista_lineas = leer_fichero(ruta_fichero)

    inventario_valido = []
    errores = []
    
    for i, linea in enumerate(lista_lineas):
        diccionario_preliminar = normalizar_linea(linea)
        es_valido, resultado = validar_datos(diccionario_preliminar)
        
        if es_valido:
            inventario_valido.append(resultado)
        else:
            errores.append(f"Liña {i+1}: '{linea}' descartada. Razón: {resultado}")
    
    # Cálculo de métricas
    num_validos = len(inventario_valido)
    num_descartados = len(errores)

    ips_unicas = set()
    responsables_distintos = set()
    
    for servidor in inventario_valido:
        ips_unicas.add(servidor['ip'])
        if servidor['responsable'] is not None:
            responsables_distintos.add(servidor['responsable'])

    ips_str = ", ".join(sorted(list(ips_unicas))) 
    
    contenido_informe = [
        f"Número de servidores válidos cargados: {num_validos}",
        f"Número de líneas descartadas (con errores): {num_descartados}",
        f"Lista de IPs únicas: {ips_str}",
        f"Responsables distintos: {len(responsables_distintos)}"
    ]
    
    # Escritura del informe
    try:
        with open("informe_servidores.txt", 'w', encoding='utf-8') as f:
            for linea in contenido_informe:
                f.write(linea + '\n')
    except Exception as e:
        print(f"Erro ao escribir o informe: {e}")

    return inventario_valido, errores, contenido_informe



print("--- Iniciando o Procesamento de Inventario ---")

inventario, errores, informe_resumen = procesar_inventario("inventario.txt")

print("\n--- INFORME RESUMO  ---")
for linea in informe_resumen:
    print(linea)

print("\n--- ERRORES ENCONTRADOS ---")
for error in errores:
    print(error)

--- Iniciando o Procesamento de Inventario ---

--- INFORME RESUMO (Consola) ---
Número de servidores válidos cargados: 6
Número de líneas descartadas (con errores): 4
Lista de IPs únicas: 10.0.0.5, 192.168.1.10, 192.168.1.20, 192.168.1.30
Responsables distintos: 3

--- ERRORES ENCONTRADOS ---
Liña 7: '; 192.168.1.40 ; linux ; sala 3 ; María' descartada. Razón: ERROR_NOMBRE_INVALIDO: O nome está baleiro.
Liña 8: 'srv-mail02 ; 300.168.1.50 ; windows ; sala 3 ; Pedro' descartada. Razón: ERROR_IP_INVALIDA: O enderezo IP non é válido.
Liña 9: 'srv-extra01 ; 192.168.1.60 ; solaris ; sala 4 ; Paula' descartada. Razón: ERROR_SISTEMA_INVALIDO: Sistema operativo non permitido.
Liña 10: 'srv-monitor01 ; 10.0.0.6' descartada. Razón: ERROR_SISTEMA_INVALIDO: Sistema operativo non permitido.
